In [21]:
import os
import re
import math
import shutil
import subprocess

from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from matplotlib.dates import DateFormatter
from dateutil.tz import tzutc, tzlocal
from scipy import stats
from scipy.optimize import brentq, curve_fit, fsolve

In [22]:
%load_ext autoreload
%autoreload 2

import official_GSSHA_Python_functions as gf

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:

# Get the current working directory
cwd = os.getcwd()
script_dir = Path.cwd()  # Use  if you're in Jupyter or interactive session

GSSHA_prj_name = "Waiahole_FIM_MODEL"
xys_tsf_iteration_file_name = "Waiahole_FIM_Model_ITERATION_file"
model_dir = script_dir / GSSHA_prj_name
GSSHA_executables = script_dir / 'GSSHA_applications'


### LOW FLOW ###
FLOW_TEST = [50]
# low_flow_time = 500
# high_flow_time = 360


for flow in FLOW_TEST:
    new_value = float(flow)
    
    gf.replace_value_in_gssha_file(read_dir=model_dir , gssha_sample_file = xys_tsf_iteration_file_name + "_bc.tsf", save_filename = GSSHA_prj_name + "_bc.tsf", old_value = "XX",  new_value = new_value)

    gf.replace_value_in_gssha_file(read_dir= model_dir, gssha_sample_file = xys_tsf_iteration_file_name + ".xys", save_filename = GSSHA_prj_name + ".xys", old_value = "XX", new_value = new_value)

    df_prj = gf.read_prj_file(GSSHA_prj_name +".prj", prj_folder_path = model_dir)
  
    ows = df_prj.Value['OVERLAND_WSE'].split('\\')[:-1]
    ows.append(GSSHA_prj_name + "_flowFIMTEST_USGSgauge_" + str(flow) + ".ows")
    ows = '\\'.join(ows) + '"'

        #.otl(flow output)
    otl = df_prj.Value['OUTLET_HYDRO'].split('\\')[:-1]
    otl.append(GSSHA_prj_name + "_flowGSSHATimingTEST_streamoutlet" + str(flow) + ".otl")
    otl = '\\'.join(otl) + '"'
    df_prj.Value['OVERLAND_WSE'] = ows
    df_prj.Value['OUTLET_HYDRO'] = otl

    if flow <= 50:
        tot_time = low_flow_time
        df_prj.Value['TOT_TIME'] = tot_time
    else:
        tot_time = high_flow_time
        df_prj.Value['TOT_TIME'] = tot_time
    
    df_prj = df_prj.reset_index()
    
    gf.convert_df_to_prj(
        df_prj,
        output_folder= model_dir,
        prj_file_name=GSSHA_prj_name
    )
    
    gf.copy_gssha_apps_to_model(GSSHA_executables, model_dir)
    gf.run_gssha(model_dir, PROJECT_FILE = GSSHA_prj_name + ".prj")
    # move_and_rename_gssha_output(MODEL_DIR, RESULTS_DIR, output_description = "TEST_RUN" + first_date, extension = "otl")
    gf.cleanup_model_dir(model_dir)




C:\Users\bgorberg\AppData\Local\Temp\ipykernel_21656\3092865827.py:34: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_prj.Value['OVERLAND_WSE'] = ows
C:\Users\bgorberg\AppData\Local\Temp\ipykernel_21656\3092865827.py:35: FutureWarning: Cha

✅ Successfully wrote: C:\Users\bgorberg\Documents\GitHub\Flood_Stage_Maps\RUN_GSSHA\Waiahole_FIM_MODEL\Waiahole_FIM_MODEL.prj
✔️ GSSHA applications copied to model folder.
📦 Running GSSHA:
  Executable: C:\Users\bgorberg\Documents\GitHub\Flood_Stage_Maps\RUN_GSSHA\Waiahole_FIM_MODEL\gssha.exe
  Project File: C:\Users\bgorberg\Documents\GitHub\Flood_Stage_Maps\RUN_GSSHA\Waiahole_FIM_MODEL\Waiahole_FIM_MODEL.prj
✅ GSSHA simulation ran successfully. Check gssha_log.txt for output.
✔️ Cleaned up model folder.
